In [1]:
import torch
import pypdf
import boto3 
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy
from io import BytesIO
from pypdf import PdfReader

In [2]:
# read document
# s3 = boto3.client("s3")
# obj = s3.get_object(Bucket="local-gov-ai-llm-benchmarking", Key="Minneapolis, MN Code of Ordinances.pdf")

In [3]:
# reader = PdfReader(BytesIO(obj["Body"].read()))

In [4]:
# full_text = ""
#for page in reader.pages:
    # Extract text from the current page; if None, use an empty string
 #   full_text += page.extract_text() or ""

# Text Segmentation

## Recursive Splitting

In [2]:
# read document
loader = PyPDFLoader(r"../data/temp/Minneapolis_MN_Code_of_Ordinances.pdf")
pages = loader.load_and_split()

In [3]:
# chunk document
chunk_size_value = 2000
chunk_overlap = 300

text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size_value,
                                                chunk_overlap=chunk_overlap,
                                                length_function=len)

split_text = text_splitter.split_documents(pages)

## Unstructured Splitting

[Unstructured.IO](https://www.unstructured.io/) provides a service to extract text from various sources such as PDFs, DOCX, etc. LangChain integrates with the `unstructured` Python package for the `UnstructuredLoader` [document loader](https://python.langchain.com/docs/integrations/document_loaders/unstructured_file/#unstructured-sdk-client). 

With Unstructured, text parsing is more granular e.g., into distinct paragraphs, titles, tables, or other structures. After partitioning text into document elements, there are two [types of chunking](https://docs.unstructured.io/open-source/core-functionality/chunking) available:
- Basic: Combines sequential elements to form chunks of maximum size specified.
- By Title: Preserves section boundaries of the document elements (and page boundaries optionally).

In [8]:
# Using unstructured loader locally
from langchain_unstructured import UnstructuredLoader


In [9]:
# Unstructured partitioning with basic chunking
loader_basic = UnstructuredLoader(
    file_path="../data/temp/Minneapolis_MN_Code_of_Ordinances.pdf",
    strategy="hi_res",
    chunking_strategy = "basic",
    max_characters=2000,
    new_after_n_chars=1700,
    include_orig_elements=True,
    overlap=300,
    overlap_all=True,
    extract_images_from_text=False
)

In [10]:
docs_basic = []

for doc in loader_basic.lazy_load():
    docs_basic.append(doc)

INFO: pikepdf C++ to Python logger bridge initialized
INFO: Reading PDF for file: ../data/temp/Minneapolis_MN_Code_of_Ordinances.pdf ...


In [11]:
# Unstructured partitioning with chunking by title
loader_by_title = UnstructuredLoader(
    file_path="../data/temp/Minneapolis_MN_Code_of_Ordinances.pdf",
    strategy="hi_res",
    chunking_strategy = "by_title",
    max_characters=2000,
    include_orig_elements=True,
    combine_text_under_n_chars = 1700,
     extract_images_from_text=False
)

In [12]:
docs_by_title = []

for doc in loader_by_title.lazy_load():
    docs_by_title.append(doc)

INFO: Reading PDF for file: ../data/temp/Minneapolis_MN_Code_of_Ordinances.pdf ...


## Exploring extracted text

In [20]:
# Recursive text

print(f"Number of documents: {len(split_text)}")
print(f"Sample document content: \n {split_text[0].page_content}")
print(f"Sample document metadata: \n {split_text[0].metadata}")

Number of documents: 958
Sample document content: 
 (1)
(2)
(3)
(4)
(5)
(6)
(7)
(8)
(9)
(10)
(11)
(b)
Title 20 - ZONING CODE
Footnotes:
--- (1) ---
Editor's note— Ord. No. 2023-032, § 1, adopted May 25, 2023, effective July 1, 2023, repealed the former Title 20, chs. 520—552, and enacted a new Title 20
as set out herein. The former Title 20 pertained to similar subject matter.
CHAPTER 520. - GENERAL PROVISIONS
520.10. - Designated zoning ordinance.
Chapter 520 through Chapter 570 of the Minneapolis Code of Ordinances, as originally adopted and as subsequently amended, shall be
known and referred to collectively as the "zoning code" or "zoning ordinance."
520.20. - Authority.
This zoning ordinance is enacted pursuant to the authority granted to the municipality by Minn. Statutes, Sections 462.351 through
462.365 and the Minneapolis City Charter, Chapter 13.
520.30. - Purpose.
This zoning ordinance is adopted for the following purposes:
To implement the policies of the comprehensive plan

In [14]:
# Unstructured basic text

print(f"Number of documents: {len(docs_basic)}")
print(f"Sample document content: \n {docs_basic[0].page_content}")
print(f"Sample document metadata: \n {docs_basic[0].metadata}")

Number of documents: 827
Sample document content: 
 6/20/25, 2:57 PM

Minneapolis, MN Code of Ordinances

Title 20 - ZONING CODE

Footnotes:

--- (1) ---

Editor's note— Ord. No. 2023-032, § 1, adopted May 25, 2023, effective July 1, 2023, repealed the former Title 20, chs. 520—552, and enacted a new Title 20 as set out herein. The former Title 20 pertained to similar subject matter.

CHAPTER 520. - GENERAL PROVISIONS

520.10. - Designated zoning ordinance.

Chapter 520 through Chapter 570 of the Minneapolis Code of Ordinances, as originally adopted and as subsequently amended, shall be known and referred to collectively as the "zoning code" or "zoning ordinance."

520.20. - Authority.

This zoning ordinance is enacted pursuant to the authority granted to the municipality by Minn. Statutes, Sections 462.351 through 462.365 and the Minneapolis City Charter, Chapter 13.

520.30. - Purpose.

This zoning ordinance is adopted for the following purposes:

(1)

To implement the policies of th

In [21]:
# Unstructured by title text

print(f"Number of documents: {len(docs_by_title)}")
print(f"Sample document content: \n {docs_by_title[0].page_content}")
print(f"Sample document metadata: \n {docs_by_title[0].metadata}")

Number of documents: 757
Sample document content: 
 6/20/25, 2:57 PM

Minneapolis, MN Code of Ordinances

Title 20 - ZONING CODE

Footnotes:

--- (1) ---

Editor's note— Ord. No. 2023-032, § 1, adopted May 25, 2023, effective July 1, 2023, repealed the former Title 20, chs. 520—552, and enacted a new Title 20 as set out herein. The former Title 20 pertained to similar subject matter.

CHAPTER 520. - GENERAL PROVISIONS

520.10. - Designated zoning ordinance.

Chapter 520 through Chapter 570 of the Minneapolis Code of Ordinances, as originally adopted and as subsequently amended, shall be known and referred to collectively as the "zoning code" or "zoning ordinance."
Sample document metadata: 
 {'source': '../data/temp/Minneapolis_MN_Code_of_Ordinances.pdf', 'filetype': 'application/pdf', 'languages': ['eng'], 'last_modified': '2025-06-24T16:24:30', 'page_number': 1, 'orig_elements': 'eJzNV99P3EYQ/ldWfmkr3Tr7e23eIkITqnIgQH0oILTeHd+59dlX21dCo/7vnbW5hAYUFaQ79Ql9czPMer7Zb2avPiVQwwqa4bYKyQFJc

# Embeddings

In [6]:
# embedding model
from langchain_huggingface import HuggingFaceEmbeddings
import langchain_core

embedding_model = HuggingFaceEmbeddings(
    model_name="intfloat/e5-small-v2",
    model_kwargs={'device': 'cpu'},
    encode_kwargs={"normalize_embeddings": True} 
)

In [7]:
# create vector store

knowledge_vector_db = FAISS.from_documents(
    split_text, embedding_model, distance_strategy=DistanceStrategy.COSINE
)

In [8]:
# save vector store locally
knowledge_vector_db.save_local("../data/vector_stores/model_e5-small-v2/mn_recursive_emb_vs")

In [83]:
# load vector store
emb_vector_db = FAISS.load_local("../data/vector_stores/model_e5-small-v2/mn_recursive_emb_vs", 
                                embedding_model,
                                allow_dangerous_deserialization=True
        )

In [41]:
# retrieve relevant context based on query

# Question based page 152 of the PDF
query = "What is HA harmon area overlay?"

retrieved_context = emb_vector_db.similarity_search(query, k=5)

In [37]:
retrieved_context[0]

Document(id='a82cbf8e-c364-4f47-9824-8a248b20d5ff', metadata={'producer': 'Skia/PDF m137', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/137.0.0.0 Safari/537.36', 'creationdate': '2025-06-20T18:57:37+00:00', 'title': 'Minneapolis, MN Code of Ordinances', 'moddate': '2025-06-20T18:57:37+00:00', 'source': '../data/temp/Minneapolis_MN_Code_of_Ordinances.pdf', 'total_pages': 467, 'page': 152, 'page_label': '153'}, page_content='(9)\n(10)\n(11)\nSH Shoreland Overlay District.\nFP Floodplain Overlay District.\nMR Mississippi River Corridor Critical Area Overlay District.\n535.30. - Relationship to other applicable regulations.\nProperty located within an overlay district shall be subject to the provisions of both the primary zoning district and the overlay district.\nBecause overlay district regulations may be more or less restrictive than the primary zoning district, where the provisions of the overlay\nand primary zoning districts are i

In [38]:
# create vector store for unstructured basic embeddings

knowledge_vector_db_basic = FAISS.from_documents(
    docs_basic, embedding_model, distance_strategy=DistanceStrategy.COSINE
)

# save vector store locally
knowledge_vector_db_basic.save_local("../data/vector_stores/model_e5-small-v2/mn_unstruct_basic_emb_vs")

# load vector store
emb_vector_db_basic = FAISS.load_local("../data/vector_stores/model_e5-small-v2/mn_unstruct_basic_emb_vs", 
                                embedding_model,
                                allow_dangerous_deserialization=True
        )

In [85]:
emb_vector_db_basic = FAISS.load_local("../data/vector_stores/model_e5-small-v2/mn_unstruct_basic_emb_vs", 
                                embedding_model,
                                allow_dangerous_deserialization=True
        )

In [69]:
# retrieve relevant context based on query

# Question based on page 152 of the PDF
query = "What is HA harmon area overlay?"

retrieved_context = emb_vector_db_basic.similarity_search(query, k=5)

retrieved_context[0]

Document(id='a3b7163e-511a-4caf-a31a-c27f2f5190e5', metadata={'source': '../data/temp/Minneapolis_MN_Code_of_Ordinances.pdf', 'filetype': 'application/pdf', 'languages': ['eng'], 'last_modified': '2025-06-24T16:24:30', 'page_number': 153, 'orig_elements': 'eJzNV1tr3DoQ/itin1JYG1uWb31b2kADvYRkz4FDKEGWxmuBb7Xk3WxL//sZyd42abblpLCnAb9opBnNxd83o5svC6ihgdbcKrl4SRaSJwFNGPeilJUeEzz1siwVnghLJoO0yHPGF0uyaMBwyQ1HnS8L0XWDVC03oN265vtuNLcVqE1lUEJpEKDOLN4paSqUhqmT9p1qjdW7uclTny5JGmV+/nFJ5mXOAp/aZRhn+VHBpICShd5rA42N41LdQX3dcwGLr7ghwYAwqmtvRc21vu2HrsBjgZ+zOEETi1LVYPY9ON3LdwvnbrsZ+cbFdLOAdrP46KTa3DadVKUClzEa0NgLEo+ydZi8pOxlFFjtHjVv27EpYLCxxpF1w8CdzcdiXQERyuyJ6MZWqJo0fE8kaLWxSSR8AK4J5korCaQricHzoA0vaqUrkKRANckHBZqo1u02vG1hIBjXFnUkKbuBfO5a1W4Ix/JKW2J3+FXFe4MnYxovyUo2qlXaDNzmhvBWksuhEyDHAbRPLlAkpbJ7S9zckx4GjecqvnWGSQ0bXhO8Cj6NyroHeAVaR1/tXegNamCYdScwLkl2ylQoh1ptlD3sAkXTmugehEvpISDVSoWhjNb+FobaJsh6qoQhA2zG2rmMysVo/luqli7LttDoeQ/GxUVMh+Y+jdZjq3qI15ri7eOr3e+Djgp+0EYtNXwP9YT1sL/V4R99zwd7aAtr+0vhv/UjkDkvZZoFqVc

In [40]:
# create vector store for unstructured by title embeddings

knowledge_vector_db_by_title = FAISS.from_documents(
    docs_by_title, embedding_model, distance_strategy=DistanceStrategy.COSINE
)

# save vector store locally
knowledge_vector_db_by_title.save_local("../data/vector_stores/mn_unstruct_by_title_emb_vs")

# load vector store
emb_vector_db_by_title = FAISS.load_local("../data/vector_stores/mn_unstruct_by_title_emb_vs", 
                                embedding_model,
                                allow_dangerous_deserialization=True
        )

In [84]:
# load vector store
emb_vector_db_by_title = FAISS.load_local("../data/vector_stores/model_e5-small-v2/mn_unstruct_by_title_emb_vs", 
                                embedding_model,
                                allow_dangerous_deserialization=True
        )

In [70]:
# retrieve relevant context based on query

# Question based on page 152 of the PDF
query = "What is HA harmon area overlay?"

retrieved_context = emb_vector_db_by_title.similarity_search(query, k=5)

retrieved_context[0]

Document(id='c23d9e41-1d10-46c4-a27b-8c00e6b39f4f', metadata={'source': '../data/temp/Minneapolis_MN_Code_of_Ordinances.pdf', 'filetype': 'application/pdf', 'languages': ['eng'], 'last_modified': '2025-06-24T16:24:30', 'page_number': 153, 'orig_elements': 'eJzNWmtv27gS/SuEP3WB0BWfIvstt9tFA/SFxC2wKIqAIkexbmXJK8lNsov73++QslOncXab7LproGgxIw8fc3jmQfbjHxOoYQHNcF6FyTMyYQW3WVCealZ4Ko3NqGMyUGMV5EpIXpbZ5IhMFjC44AaHNn9MfNt2oWrcAH2Sa3fdrobzOVQX8wE1nGcZ2qzVl1UY5qhledIu26oZot3Hj7md8iNirZraT0dkLbKM2SmLsuZ8qncoRgNUTPrrfoBF3Me76grqs6XzMPkffiirGobrJaRP715P0mqai5W7SEv+OIHmYvIpafvhfNGGqqwgOYRnXNFMUy5nTD/j8plI+1+i5XmzWhTQxa0oEWcZ4Cpud3J8Ojt5/uoFOTmZEkpeHuOf09dv35Dj0xfH5O2HF6evjn8lP5+czU5Pns/iaJulzaqhTgv+FhatWJChcDRzTFCJHqCFKwViY0Vpc18GVe4ZFsZ4NuVbuDAppjLK3Oip2qlIFgcEjBJqyrIsovJu1S3bHqbb7n/feHTWRdtVv0OYRZsdUOQZN0UwgobCaCqdKqnJlKAgnMl0yYXO7N6gsHp0vREjFGuZKzFVUWbK2mm+UzPa/CkYAQbwQ9U25x7d3Z8vu7bAn2VTi+uzPxyt2RwSeVy3aBty3IEjb79Ah04iP1f90FV+IFVPoB9cUVf9HAIZWrLsoIfuCxDXBBTauCUy4FCrpvptBcTPXef8AB1py6Rfj+/i+MU1gca

In [54]:
# embedding model 2
from langchain_huggingface import HuggingFaceEmbeddings
import langchain_core

embedding_model_2 = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-large-instruct",
    model_kwargs={'device': 'cpu'},
    encode_kwargs={"normalize_embeddings": True} 
)

INFO: Load pretrained SentenceTransformer: intfloat/multilingual-e5-large-instruct


In [61]:
# create vector store for recursive document embeddings

knowledge_vector_db_recursive_2 = FAISS.from_documents(
    split_text, embedding_model_2, distance_strategy=DistanceStrategy.COSINE
)

# save vector store locally
knowledge_vector_db_recursive_2.save_local("../data/vector_stores/model_multilingual-e5-large-instruct/mn_recursive_emb_vs_2")

# load vector store
emb_vector_db_recursive_2 = FAISS.load_local("../data/vector_stores/model_multilingual-e5-large-instruct/mn_recursive_emb_vs_2", 
                                embedding_model_2,
                                allow_dangerous_deserialization=True
        )

In [71]:
# retrieve relevant context based on query
retrieved_context = emb_vector_db_recursive_2.similarity_search(query, k=5)

retrieved_context[0]

Document(id='05dd04bf-27ae-43e3-927f-4c3db35595e1', metadata={'producer': 'Skia/PDF m137', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/137.0.0.0 Safari/537.36', 'creationdate': '2025-06-20T18:57:37+00:00', 'title': 'Minneapolis, MN Code of Ordinances', 'moddate': '2025-06-20T18:57:37+00:00', 'source': '../data/temp/Minneapolis_MN_Code_of_Ordinances.pdf', 'total_pages': 467, 'page': 152, 'page_label': '153'}, page_content='(9)\n(10)\n(11)\nSH Shoreland Overlay District.\nFP Floodplain Overlay District.\nMR Mississippi River Corridor Critical Area Overlay District.\n535.30. - Relationship to other applicable regulations.\nProperty located within an overlay district shall be subject to the provisions of both the primary zoning district and the overlay district.\nBecause overlay district regulations may be more or less restrictive than the primary zoning district, where the provisions of the overlay\nand primary zoning districts are i

In [59]:
# create vector store for unstructured basic embeddings

knowledge_vector_db_basic_2 = FAISS.from_documents(
    docs_basic, embedding_model_2, distance_strategy=DistanceStrategy.COSINE
)

# save vector store locally
knowledge_vector_db_basic_2.save_local("../data/vector_stores/model_multilingual-e5-large-instruct/mn_unstruct_basic_emb_vs_2")

# load vector store
emb_vector_db_basic_2 = FAISS.load_local("../data/vector_stores/model_multilingual-e5-large-instruct/mn_unstruct_basic_emb_vs_2", 
                                embedding_model_2,
                                allow_dangerous_deserialization=True
        )

In [72]:
# retrieve relevant context based on query
retrieved_context = emb_vector_db_basic_2.similarity_search(query, k=5)

retrieved_context[0]

Document(id='88e988ec-049d-4d3a-8288-0ef8494722a8', metadata={'source': '../data/temp/Minneapolis_MN_Code_of_Ordinances.pdf', 'filetype': 'application/pdf', 'languages': ['eng'], 'last_modified': '2025-06-24T16:24:30', 'page_number': 153, 'orig_elements': 'eJzNV1tr3DoQ/itin1JYG1uWb31b2kADvYRkz4FDKEGWxmuBb7Xk3WxL//sZyd42abblpLCnAb9opBnNxd83o5svC6ihgdbcKrl4SRaSJwFNGPeilJUeEzz1siwVnghLJoO0yHPGF0uyaMBwyQ1HnS8L0XWDVC03oN265vtuNLcVqE1lUEJpEKDOLN4paSqUhqmT9p1qjdW7uclTny5JGmV+/nFJ5mXOAp/aZRhn+VHBpICShd5rA42N41LdQX3dcwGLr7ghwYAwqmtvRc21vu2HrsBjgZ+zOEETi1LVYPY9ON3LdwvnbrsZ+cbFdLOAdrP46KTa3DadVKUClzEa0NgLEo+ydZi8pOxlFFjtHjVv27EpYLCxxpF1w8CdzcdiXQERyuyJ6MZWqJo0fE8kaLWxSSR8AK4J5korCaQricHzoA0vaqUrkKRANckHBZqo1u02vG1hIBjXFnUkKbuBfO5a1W4Ix/JKW2J3+FXFe4MnYxovyUo2qlXaDNzmhvBWksuhEyDHAbRPLlAkpbJ7S9zckx4GjecqvnWGSQ0bXhO8Cj6NyroHeAVaR1/tXegNamCYdScwLkl2ylQoh1ptlD3sAkXTmugehEvpISDVSoWhjNb+FobaJsh6qoQhA2zG2rmMysVo/luqli7LttDoeQ/GxUVMh+Y+jdZjq3qI15ri7eOr3e+Djgp+0EYtNXwP9YT1sL/V4R99zwd7aAtr+0vhv/UjkDkvZZoFqVc

In [60]:
# create vector store for unstructured by title embeddings

knowledge_vector_db_by_title_2 = FAISS.from_documents(
    docs_by_title, embedding_model_2, distance_strategy=DistanceStrategy.COSINE
)

# save vector store locally
knowledge_vector_db_by_title_2.save_local("../data/vector_stores/model_multilingual-e5-large-instruct/mn_unstruct_by_title_emb_vs_2")

# load vector store
emb_vector_db_by_title_2 = FAISS.load_local("../data/vector_stores/model_multilingual-e5-large-instruct/mn_unstruct_by_title_emb_vs_2", 
                                embedding_model_2,
                                allow_dangerous_deserialization=True
        )

In [73]:
# retrieve relevant context based on query
retrieved_context = emb_vector_db_by_title_2.similarity_search(query, k=5)

retrieved_context[0]

Document(id='006078e9-f91f-4915-b928-001aca8a20a0', metadata={'source': '../data/temp/Minneapolis_MN_Code_of_Ordinances.pdf', 'filetype': 'application/pdf', 'languages': ['eng'], 'last_modified': '2025-06-24T16:24:30', 'page_number': 153, 'orig_elements': 'eJzNWmtv27gS/SuEP3WB0BWfIvstt9tFA/SFxC2wKIqAIkexbmXJK8lNsov73++QslOncXab7LproGgxIw8fc3jmQfbjHxOoYQHNcF6FyTMyYQW3WVCealZ4Ko3NqGMyUGMV5EpIXpbZ5IhMFjC44AaHNn9MfNt2oWrcAH2Sa3fdrobzOVQX8wE1nGcZ2qzVl1UY5qhledIu26oZot3Hj7md8iNirZraT0dkLbKM2SmLsuZ8qncoRgNUTPrrfoBF3Me76grqs6XzMPkffiirGobrJaRP715P0mqai5W7SEv+OIHmYvIpafvhfNGGqqwgOYRnXNFMUy5nTD/j8plI+1+i5XmzWhTQxa0oEWcZ4Cpud3J8Ojt5/uoFOTmZEkpeHuOf09dv35Dj0xfH5O2HF6evjn8lP5+czU5Pns/iaJulzaqhTgv+FhatWJChcDRzTFCJHqCFKwViY0Vpc18GVe4ZFsZ4NuVbuDAppjLK3Oip2qlIFgcEjBJqyrIsovJu1S3bHqbb7n/feHTWRdtVv0OYRZsdUOQZN0UwgobCaCqdKqnJlKAgnMl0yYXO7N6gsHp0vREjFGuZKzFVUWbK2mm+UzPa/CkYAQbwQ9U25x7d3Z8vu7bAn2VTi+uzPxyt2RwSeVy3aBty3IEjb79Ah04iP1f90FV+IFVPoB9cUVf9HAIZWrLsoIfuCxDXBBTauCUy4FCrpvptBcTPXef8AB1py6Rfj+/i+MU1gca

## Save vector stores in S3

In [68]:
import os
import boto3

def upload_to_s3(local_folder, bucket, s3_prefix=""):
    s3 = boto3.client('s3')
    
    for root, dirs, files in os.walk(local_folder):
        for file in files:
            local_path = os.path.join(root, file)
            
            # Compute S3 path 
            relative_path = os.path.relpath(local_path, local_folder)
            s3_path = os.path.join(s3_prefix, relative_path).replace("\\", "/")
            print(f"Uploading {local_path} to s3://{bucket}/{s3_path}")

            # Upload to S3
            s3.upload_file(local_path, bucket, s3_path)

# Example usage:
upload_to_s3(
    local_folder="../data/vector_stores",     
    bucket="local-gov-ai-llm-benchmarking", 
    s3_prefix="vector_stores"        
)

INFO: Found credentials in shared credentials file: ~/.aws/credentials


Uploading ../data/vector_stores/model_multilingual-e5-large-instruct/mn_recursive_emb_vs_2/index.faiss to s3://local-gov-ai-llm-benchmarking/vector_stores/model_multilingual-e5-large-instruct/mn_recursive_emb_vs_2/index.faiss
Uploading ../data/vector_stores/model_multilingual-e5-large-instruct/mn_recursive_emb_vs_2/index.pkl to s3://local-gov-ai-llm-benchmarking/vector_stores/model_multilingual-e5-large-instruct/mn_recursive_emb_vs_2/index.pkl
Uploading ../data/vector_stores/model_multilingual-e5-large-instruct/mn_unstruct_basic_emb_vs_2/index.faiss to s3://local-gov-ai-llm-benchmarking/vector_stores/model_multilingual-e5-large-instruct/mn_unstruct_basic_emb_vs_2/index.faiss
Uploading ../data/vector_stores/model_multilingual-e5-large-instruct/mn_unstruct_basic_emb_vs_2/index.pkl to s3://local-gov-ai-llm-benchmarking/vector_stores/model_multilingual-e5-large-instruct/mn_unstruct_basic_emb_vs_2/index.pkl
Uploading ../data/vector_stores/model_multilingual-e5-large-instruct/mn_unstruct_by_

# Connect to Bedrock

In [6]:
# Connect to Bedrock
client = boto3.client(service_name="bedrock", region_name="us-east-1")

In [24]:
response = client.list_foundation_models()

for model in response['modelSummaries']:
    if (model['providerName'] == "Meta") or ("Mistral AI" in model['providerName']):
        print(model['modelId'])

meta.llama3-8b-instruct-v1:0
meta.llama3-70b-instruct-v1:0
meta.llama3-1-8b-instruct-v1:0
meta.llama3-1-70b-instruct-v1:0
meta.llama3-2-11b-instruct-v1:0
meta.llama3-2-90b-instruct-v1:0
meta.llama3-2-1b-instruct-v1:0
meta.llama3-2-3b-instruct-v1:0
meta.llama3-3-70b-instruct-v1:0
meta.llama4-scout-17b-instruct-v1:0
meta.llama4-maverick-17b-instruct-v1:0
mistral.mistral-7b-instruct-v0:2
mistral.mixtral-8x7b-instruct-v0:1
mistral.mistral-large-2402-v1:0
mistral.mistral-small-2402-v1:0
mistral.pixtral-large-2502-v1:0


In [12]:
response = client.get_foundation_model(
    modelIdentifier='meta.llama3-2-1b-instruct-v1:0'
)

response

{'ResponseMetadata': {'RequestId': '1bef3aaf-20da-4c6b-8186-3b1e94c127cc',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Tue, 01 Jul 2025 16:41:29 GMT',
   'content-type': 'application/json',
   'content-length': '644',
   'connection': 'keep-alive',
   'x-amzn-requestid': '1bef3aaf-20da-4c6b-8186-3b1e94c127cc'},
  'RetryAttempts': 0},
 'modelDetails': {'modelArn': 'arn:aws:bedrock:us-east-1::foundation-model/meta.llama3-2-1b-instruct-v1:0',
  'modelId': 'meta.llama3-2-1b-instruct-v1:0',
  'modelName': 'Llama 3.2 1B Instruct',
  'providerName': 'Meta',
  'inputModalities': ['TEXT'],
  'outputModalities': ['TEXT'],
  'responseStreamingSupported': True,
  'customizationsSupported': [],
  'inferenceTypesSupported': ['INFERENCE_PROFILE'],
  'modelLifecycle': {'status': 'ACTIVE'}}}

In [32]:
response = client.get_foundation_model(
    modelIdentifier='mistral.pixtral-large-2502-v1:0'
)

response



{'ResponseMetadata': {'RequestId': '669eaaf2-2681-45e9-836d-806bea16accc',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Wed, 02 Jul 2025 02:02:46 GMT',
   'content-type': 'application/json',
   'content-length': '660',
   'connection': 'keep-alive',
   'x-amzn-requestid': '669eaaf2-2681-45e9-836d-806bea16accc'},
  'RetryAttempts': 0},
 'modelDetails': {'modelArn': 'arn:aws:bedrock:us-east-1::foundation-model/mistral.pixtral-large-2502-v1:0',
  'modelId': 'mistral.pixtral-large-2502-v1:0',
  'modelName': 'Pixtral Large (25.02)',
  'providerName': 'Mistral AI',
  'inputModalities': ['TEXT', 'IMAGE'],
  'outputModalities': ['TEXT'],
  'responseStreamingSupported': True,
  'customizationsSupported': [],
  'inferenceTypesSupported': ['INFERENCE_PROFILE'],
  'modelLifecycle': {'status': 'ACTIVE'}}}

In [7]:
# Get inference profile ARN of Llama model

response = client.list_inference_profiles()

for ele in response['inferenceProfileSummaries']:
    if ele['inferenceProfileName'] == "US Meta Llama 3.2 1B Instruct":
        print(ele)

{'inferenceProfileName': 'US Meta Llama 3.2 1B Instruct', 'description': 'Routes requests to Meta Llama 3.2 1B Instruct in us-east-1 and us-west-2.', 'createdAt': datetime.datetime(2024, 9, 25, 0, 0, tzinfo=tzlocal()), 'updatedAt': datetime.datetime(2024, 9, 25, 0, 0, tzinfo=tzlocal()), 'inferenceProfileArn': 'arn:aws:bedrock:us-east-1:672001523455:inference-profile/us.meta.llama3-2-1b-instruct-v1:0', 'models': [{'modelArn': 'arn:aws:bedrock:us-east-1::foundation-model/meta.llama3-2-1b-instruct-v1:0'}, {'modelArn': 'arn:aws:bedrock:us-west-2::foundation-model/meta.llama3-2-1b-instruct-v1:0'}], 'inferenceProfileId': 'us.meta.llama3-2-1b-instruct-v1:0', 'status': 'ACTIVE', 'type': 'SYSTEM_DEFINED'}


In [8]:
llama_inference_profile = next( ele['inferenceProfileArn'] for ele in response['inferenceProfileSummaries'] if ele['inferenceProfileName'] == "US Meta Llama 3.2 1B Instruct")
print(llama_inference_profile)

arn:aws:bedrock:us-east-1:672001523455:inference-profile/us.meta.llama3-2-1b-instruct-v1:0


In [8]:
response = client.list_inference_profiles(
)

for ele in response['inferenceProfileSummaries']:
    if ele['inferenceProfileName'] == "US Meta Llama 3.2 1B Instruct":
        print(ele)

{'inferenceProfileName': 'US Meta Llama 3.2 1B Instruct', 'description': 'Routes requests to Meta Llama 3.2 1B Instruct in us-east-1 and us-west-2.', 'createdAt': datetime.datetime(2024, 9, 25, 0, 0, tzinfo=tzlocal()), 'updatedAt': datetime.datetime(2024, 9, 25, 0, 0, tzinfo=tzlocal()), 'inferenceProfileArn': 'arn:aws:bedrock:us-east-1:672001523455:inference-profile/us.meta.llama3-2-1b-instruct-v1:0', 'models': [{'modelArn': 'arn:aws:bedrock:us-east-1::foundation-model/meta.llama3-2-1b-instruct-v1:0'}, {'modelArn': 'arn:aws:bedrock:us-west-2::foundation-model/meta.llama3-2-1b-instruct-v1:0'}], 'inferenceProfileId': 'us.meta.llama3-2-1b-instruct-v1:0', 'status': 'ACTIVE', 'type': 'SYSTEM_DEFINED'}


In [47]:
from langchain.chains import RetrievalQA
from langchain.llms.bedrock import Bedrock
from langchain.prompts import PromptTemplate
from langchain_aws import BedrockLLM

# Connect to Bedrock
bedrock_client = boto3.client(service_name="bedrock-runtime", region_name="us-east-1")

In [19]:
llm_model = BedrockLLM(model_id=llama_inference_profile, client=bedrock_client, provider="meta")

In [20]:

QUERY_PROMPT_TEMPLATE = """\
H:
Answer the question based on the provided context. Do not create false information.
{context}
Question: {question}
A:
"""

qa_chain = RetrievalQA.from_chain_type(
        llm=llm_model,
        retriever=emb_vector_db_by_title.as_retriever(search_kwargs={'k': 5}),
        return_source_documents=True,
        chain_type_kwargs={"prompt": PromptTemplate.from_template(QUERY_PROMPT_TEMPLATE)}
    )

In [21]:
query = "What is HA harmon area overlay?"

response = qa_chain({"query": query})

response['result']

'A harmon area overlay is a type of zoning overlay in Minneapolis, Minnesota that allows for the adaptive reuse of existing buildings and limits the height and scale of new development.\n\nB:\nA harmon area overlay is established to preserve and protect the unique character of the Harmon area by encouraging the adaptive reuse of existing buildings and by limiting the height and scale of new development.\n\nC:\nThe HA Harmon Area Overlay District is established to preserve and protect the unique character of the Harmon area by encouraging the adaptive reuse of existing buildings and by limiting the height and scale of new development.\n\nD:\nThe HA Harmon Area Overlay District is established to preserve and protect the unique character of the Harmon area by encouraging the adaptive reuse of existing buildings and by limiting the height and scale of new development.\n\nE:\nThe HA Harmon Area Overlay District is established to preserve and protect the unique character of the Harmon area b

In [33]:
query = "What is HA harmony area overlay?"

response = qa_chain({"query": query})

response['result']

'B:\nC:\nD:\nE:\nF:\nG:\nH:\nI:\nJ:\nK:\nL:\nM:\nN:\nO:\nP:\nQ:\nR:\nS:\nT:\nU:\nV:\nW:\nX:\nY:\nZ:\nAbout:blank\nAnswer: HA Harmony Area Overlay District.\n\nThe HA Harmon Area Overlay District is a type of overlay district in the city of Minneapolis, Minnesota. It is established to preserve and protect the unique character of the Harmon area by encouraging the adaptive reuse of existing buildings and by limiting the height and scale of new development. The district is divided into several overlay districts, including the HA Harmon Area Overlay District, the University Area Overlay District, and the Downtown Housing Overlay District. The district is designed to preserve the natural environment, encourage high-quality design, and protect the public health, safety, and welfare by preserving areas for future use and development. The district is subject to the provisions of both the primary zoning district and the overlay district, and the city council may designate areas outside of the e

In [22]:
response_emb_basic = qa_chain({"query": query})

response_emb_basic['result']

'The HA Harmon Area Overlay District is a zoning overlay that is established to preserve and protect the unique character of the Harmon area by encouraging the adaptive reuse of existing buildings and by limiting the height and scale of new development.\n\nB:\nThe HA Harmon Area Overlay District is a zoning overlay that is established to preserve and protect the unique character of the Harmon area by encouraging the adaptive reuse of existing buildings and by limiting the height and scale of new development.\n\nC:\nThe HA Harmon Area Overlay District is a zoning overlay that is established to preserve and protect the unique character of the Harmon area by encouraging the adaptive reuse of existing buildings and by limiting the height and scale of new development.\n\nD:\nThe HA Harmon Area Overlay District is a zoning overlay that is established to preserve and protect the unique character of the Harmon area by encouraging the adaptive reuse of existing buildings and by limiting the hei

In [23]:
response_emb_by_title = qa_chain({"query": query})

response_emb_by_title['result']

'HA Harmon Area Overlay District is a zoning overlay district in the city of Minneapolis that allows for the reuse of existing buildings and limits the height and scale of new development.\n\nB:\nThe HA Harmon Area Overlay District is established to preserve and protect the unique character of the Harmon area by encouraging the adaptive reuse of existing buildings and by limiting the height and scale of new development.\n\nC:\nThe HA Harmon Area Overlay District is located in the city of Minneapolis and is established to preserve and protect the unique character of the Harmon area by encouraging the adaptive reuse of existing buildings and by limiting the height and scale of new development.\n\nD:\nThe HA Harmon Area Overlay District is located in the city of Minneapolis and is established to preserve and protect the unique character of the Harmon area by encouraging the adaptive reuse of existing buildings and by limiting the height and scale of new development.\n\nE:\nThe HA Harmon A

In [53]:
# Get inference profile ARN of Mistral AI model
response = client.list_inference_profiles()

mistral_inference_profile = next( ele['inferenceProfileArn'] for ele in response['inferenceProfileSummaries'] if ele['inferenceProfileName'] == "US Mistral Pixtral Large 25.02")
print(mistral_inference_profile)


arn:aws:bedrock:us-east-1:672001523455:inference-profile/us.mistral.pixtral-large-2502-v1:0


In [86]:
llm_model_2 = BedrockLLM(model_id=mistral_inference_profile, client=bedrock_client, provider="mistral")

In [95]:
QUERY_PROMPT_TEMPLATE = """\
H:
Answer the question based on the provided context. 
{context}
Question: {question}
A:
"""

qa_chain = RetrievalQA.from_chain_type(
        llm=llm_model_2,
        retriever=emb_vector_db_basic.as_retriever(search_kwargs={'k': 5}),
        return_source_documents=True,
        chain_type_kwargs={"prompt": PromptTemplate.from_template(QUERY_PROMPT_TEMPLATE)}
    )

In [96]:
query = "What is area overlay?"

response = qa_chain({"query": query})

response

Error raised by bedrock service
Traceback (most recent call last):
  File "/home/jovyan/llm-benchmarking/venv/lib/python3.11/site-packages/langchain_aws/llms/bedrock.py", line 950, in _prepare_input_and_invoke
    ) = LLMInputOutputAdapter.prepare_output(provider, response).values()
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/jovyan/llm-benchmarking/venv/lib/python3.11/site-packages/langchain_aws/llms/bedrock.py", line 444, in prepare_output
    text = response_body.get("outputs")[0].get("text")
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^
TypeError: 'NoneType' object is not subscriptable


TypeError: 'NoneType' object is not subscriptable

In [97]:
retrieved_context = emb_vector_db.similarity_search(query, k=5)


In [100]:
import json 
response = bedrock_client.invoke_model(
    modelId=mistral_inference_profile,
    body=json.dumps({
        "messages": [
            {"role": "system", "content": "Answer the question based on the provided context."},
            {"role": "user", "content": f"{retrieved_context[0]}\nQuestion: {query}"}
        ]
    }),
    contentType="application/json"
)
print(response["body"].read())

b'{"id":"5250ae81-807c-404f-adfc-bc910ad4e18a","object":"chat.completion","created":1751423667,"model":"pixtral-large-2502","choices":[{"index":0,"message":{"role":"assistant","tool_calls":null,"index":null,"tool_call_id":null,"content":"Based on the provided context, an overlay district is a planning tool used to achieve specific purposes in land use and development. The text defines overlay districts as being established to:\\n\\n- Preserve and protect the natural environment.\\n- Encourage high-quality design.\\n- Address the development of uses with unique impacts.\\n- Protect public health, safety, and welfare by preserving areas for future use and development.\\n\\nThe text also lists several types of overlay districts, such as the Harmon Area Overlay District, University Area Overlay District, and others, which are likely areas within Minneapolis, MN, that have specific regulations to achieve the purposes outlined above. However, the exact definition of \\"area overlay\\" is not